In [ ]:
"""
StreamingLLM rolling KV cache implementation.

Maintains K sink token KV entries permanently plus a sliding window
of W recent token KV entries, with position re-indexing for RoPE models.
"""

In [1]:
import torch 
from dataclasses import dataclass 
from typing import Optional, Tuple, Union, List, Dict, Any

In [2]:
@dataclass 
class StreamingKVCache:
    """
    StreamingKVCache maintains a rolling KV cache for transformer models, allowing for efficient attention computation over a sliding window of tokens.
    Args :
    num_sink_tokens (int) : k, number of inital tokens held permanently in the sink
    window_size (int) : w, number of recent tokens held in the sliding window
    num_layers (int) : number of transformer layers
    num_heads (int) : number of attention heads
    head_dim (int) : dimension of each attention head
    dtype : Tensor dtype for the KV cache tensors (fp16or bf16 for production)
    device  : Device for the KV cache tensors (cuda or cpu)
    """
    num_sink_tokens : int 
    window_size : int
    num_layers : int
    num_heads : int
    head_dim : int
    dtype : torch.dtype = torch.float16
    device : torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def __post_init__(self):
        # Sink cache  : permanently holds KV for the first k tokens
        # shape : [num_layers, 2, num_heads, num_sink_tokens, head_dim]
        # dim 1 : 0 for K, 1 for V
        self.sink_cache = torch.zeros(
            self.num_layers , 2, self.num_heads, self.num_sink_tokens, self.head_dim, dtype = self.dtype, device = self.device
        )
        # window cache : rolling buffer for the most recent w tokens
        self.window_cache = torch.zeros(
            self.num_layers, 2 , self.num_heads, self.window_size, self.head_dim, dtype = self.dtype, device = self.device
        )
        self.sink_filled = 0  # number of tokens currently in the sink cache
        self.window_len = 0 # current number of tokens in the window cache
        self.total_seen = 0 # total number of tokens processed (for position indexing)
    
    def update_cache(self, layer_idx : int, new_keys : torch.Tensor,
        new_values : torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """ 
        Add one token's KV to the cache, evicting the oldest window token if the window is full .
        Returns (keys, values, position_ids) tensors for the current attention computation.

        Args : 
        keys : [num_heads, head_dim] tensor of the new token's keys for the given layer
        values : [num_heads, head_dim] tensor of the new token's values for the given layer
        layer_idx : int, index of the transformer layer

        Returns :
        updated_keys : [total_tokens, num_heads, head_dim] tensor of keys for attention
        updated_values : [total_tokens, num_heads, head_dim] tensor of values for attention
        position_ids : [total_tokens] tensor of position IDs for RoPE
        """
        # ensure input tensors have correct shape
        assert new_keys.shape == (self.num_heads, self.head_dim)
        assert new_values.shape == (self.num_heads, self.head_dim)

        # strip the token dimension and add batch dimension for concatenation
        k = new_keys.squeeze(0)  # [1, num_heads, head_dim] #squeeze - removes dimensions of size 1 , 0 is the token dimension
        v = new_values.squeeze(0)  # [1, num_heads, head_dim]

        # fill sink slots first (first k tokens of the sequence)

        if self.sink_filled < self.num_sink_tokens:
            idx = self.sink_filled
            self.sink_cache[layer_idx, 0, :, idx, :]= k # keys to sink cache with the shape of [num_heads, head_dim]
            self.sink_cache[layer_idx, 1, :, idx, :]= v # values to sink cache
            self.sink_filled += 1
        else : 
            # slide the window cache : evict oldest, append new at the end.tuple
            if self.window_len < self.window_size:
                # window not yet full, just append
                self.window_cache[layer_idx, 0, :, self.window_len, :]= k
                self.window_cache[layer_idx, 1, :, self.window_len, :]= v
                self.window_len += 1
            else :
                # window full : roll left by 1 , place new token at end , in prod, use a circular buffer index instead of roll.tuple
                self.window_cache[layer_idx,0] = torch.roll(
                    self.window_cache[layer_idx,0], shifts=-1, dims=1
                ) # roll - shifts the elements of the tensor along the specified dimension (dims) by the given number of positions (shifts). Here, it rolls the keys in the window cache to the left by 1 position, effectively evicting the oldest token's keys.
                self.window_cache[layer_idx,1] = torch.roll(
                    self.window_cache[layer_idx,1], shifts=-1, dims=1)
                # shift : -1 means roll to the left, dims=1 means roll along the token dimension of the window cache
                self.window_cache[layer_idx, 0, :, -1, :]= k
                self.window_cache[layer_idx, 1, :, -1, :]= v

            self.total_seen += 1   
            current_len = self.sink_filled + self.window_len

            #assemble full KV for attention: [num_heads, current_len, head_dim]
            # concatenate sink and window caches along the token dimension
            k_sink = self.sink_cache[layer_idx, 0, :, :self.sink_filled, :]
            v_sink = self.sink_cache[layer_idx, 1, :, :self.sink_filled, :]
            k_window = self.window_cache[layer_idx, 0, :, :self.window_len, :]
            v_window = self.window_cache[layer_idx, 1, :, :self.window_len, :]

            keys = torch.cat([k_sink, k_window], dim = 1) # [num_heads, current_len, head_dim] where current_len = sink_filled + window_len , torch.cat concatenates the sink and window caches along the token dimension (dim=1), where dim 1 means the token dimension. The resulting keys tensor has shape [num_heads, current_len, head_dim], where current_len is the total number of tokens currently in the cache (sink + window).
            values = torch.cat([v_sink, v_window], dim = 1) # [num_heads, current_len, head_dim] same concatenation for values

            # re-index positions for RoPE : position ids are just the token indices in the original sequence, which is total_seen - current_len to total_seen - 1
            sink_pos = torch.arange(self.sink_filled, device = self.device) # position ids for sink tokens : [0, 1, ..., sink_filled - 1]
            window_pos = torch.arange( self.sink_filled, self.sink_filled + self.window_len,device =self.device) # position ids for window tokens : [sink_filled, sink_filled + 1, ..., sink_filled + window_len - 1]
            position_ids = torch.cat([sink_pos, window_pos]) # [current_len] concatenated position ids for sink and window tokens
            return keys, values, position_ids

In [3]:
def streaming_generate( model, input_ids : torch.Tensor, max_new_tokens : int = 1000, num_sink_tokens : int = 4, window_size : int = 1020) -> torch.Tensor :
    """
    Autoregressive generation using a StreamingLLM rolling KV cache.
    Works for sequences of any length without quality degradation.
    
    Args:
        model:            A causal LM with RoPE position encodings.
        input_ids:        [1, prompt_len] initial prompt.
        max_new_tokens:   How many tokens to generate.
        num_sink_tokens:  K, number of sink slots (default 4).
        window_size:      W, sliding window size (default 1020).
    
    Returns:
        generated: [1, max_new_tokens] newly generated token ids.
    """
    cache = StreamingKVCache(
        num_sink_tokens=num_sink_tokens,
        window_size=window_size,
        num_layers=model.config.n_layer,
        num_heads=model.config.n_head,
        head_dim=model.config.n_embd // model.config.n_head,
        dtype=model.dtype,
        device=input_ids.device
    )

    generated = []
    # process prompt tokens first to fill the sink and window caches
    cur_ids = input_ids
    past_kvs = None

    for step in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(cur_ids, past_key_values=past_kvs, use_cache=True)
            logits = outputs.logits
            past_kvs = outputs.past_key_values
            generated.append(logits[:, -1, :].argmax(dim=-1, keepdim=True))  # greedy decoding , logits[:, -1, :] is the logits for the last generated token, argmax selects the token with the highest probability
            cur_ids = generated[-1]  # next input is the last generated token
            # NOTE: In a full implementation, pass cache's (keys, values, position_ids)
        # back into model() at each step using a custom attention forward pass.
        # The above is pseudocode structure; the actual integration depends on
        # whether you patch model.forward() or use HuggingFace's cache API.
    return torch.cat(generated, dim=1)  # [1, max_new_tokens]